In [1]:
import os
# 选择GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
os.chdir('/home/zheling/Character_Recognition')

print("当前工作目录：", os.getcwd())


当前工作目录： /home/zheling/Character_Recognition


In [2]:
import struct
import math
import time
import json
import itertools
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import iisignature


In [3]:
def get_gpu_memory_usage_mb():
    try:
        if torch.cuda.is_available():
            return torch.cuda.memory_allocated() / (1024 * 1024)
        return 0
    except Exception:
        return 0


In [4]:
def _concat_points(strokes):
    if len(strokes) == 0:
        return np.zeros((0, 2), dtype=np.float32), []
    idx = []
    all_pts = []
    start = 0
    for s in strokes:
        s = np.asarray(s, dtype=np.float32)
        L = len(s)
        if L == 0: continue
        all_pts.append(s)
        idx.append((start, start + L))
        start += L
    if not all_pts:
        return np.zeros((0, 2), dtype=np.float32), []
    return np.concatenate(all_pts, axis=0), idx

def _rotate_pts_about_anchor(P, theta, anchor):
    c, s = np.cos(theta), np.sin(theta)
    xy = P[:, :2]
    anc_xy = anchor[:2]
    D = xy - anc_xy
    x_new = c * D[:, 0] - s * D[:, 1]
    y_new = s * D[:, 0] + c * D[:, 1]
    R = np.stack([x_new, y_new], axis=1) + anc_xy
    if P.shape[1] > 2:
        return np.concatenate([R, P[:, 2:]], axis=1)
    return R

def apply_hanging_on_strokes(strokes, mode="SC"):
    if (mode is None) or (mode == "None") or (len(strokes) == 0): return strokes
    P, spans = _concat_points(strokes)
    if P.shape[0] < 2: return strokes
    first_pt = P[0]
    if mode == "SC":
        center = P[:, :2].mean(axis=0)
        v = center - first_pt[:2]
        angle = np.arctan2(v[1], v[0]) + np.pi / 2.0
        P2 = _rotate_pts_about_anchor(P, -angle, first_pt)
        return [P2[s:e] for (s, e) in spans]
    elif mode == "SE":
        last_pt = P[-1]
        v = last_pt[:2] - first_pt[:2]
        angle = np.arctan2(v[1], v[0]) + np.pi / 2.0
        P2 = _rotate_pts_about_anchor(P, -angle, first_pt)
        return [P2[s:e] for (s, e) in spans]
    elif mode == "ASE":
        out = []
        for s in strokes:
            s_arr = np.asarray(s, dtype=np.float32)
            if len(s_arr) < 2:
                out.append(s_arr)
                continue
            start = s_arr[0]
            end = s_arr[-1]
            v = end[:2] - start[:2]
            angle = np.arctan2(v[1], v[0]) + np.pi / 2.0
            s_rot = _rotate_pts_about_anchor(s_arr, -angle, start)
            out.append(s_rot)
        return out
    return strokes

def read_pot_file(filename):
    if not os.path.exists(filename): return [], ""
    raw_strokes = []
    with open(filename, "rb") as f:
        while True:
            sample_size = f.read(2)
            if sample_size == b'': break
            dword_code = f.read(2)
            if len(dword_code) < 2: break
            if dword_code[0] != 0:
                dword_code = bytes((dword_code[1], dword_code[0]))
            tag_code = struct.unpack(">H", dword_code)[0]
            f.read(2)
            try: tag = struct.pack('>H', tag_code).decode("gbk")[0]
            except Exception: tag = str(tag_code)
            f.read(2)

            strokes_samples, stroke_samples = [], []
            while True:
                bx = f.read(2)
                by = f.read(2)
                if bx == b'' or by == b'':
                    if stroke_samples: strokes_samples.append(stroke_samples); stroke_samples = []
                    break
                if bx == b'\xff\xff' and by == b'\xff\xff':
                    if stroke_samples: strokes_samples.append(stroke_samples); stroke_samples = []
                    break
                if bx == b'\xff\xff' and by == b'\x00\x00':
                    strokes_samples.append(stroke_samples)
                    stroke_samples = []
                    continue
                x = struct.unpack("<H", bx)[0]
                y = struct.unpack("<H", by)[0]
                stroke_samples.append((x, y))
            raw_strokes.append(strokes_samples)
    return raw_strokes, tag

def normalize_strokes(strokes, coord_range=1.0):
    if not strokes: return strokes
    all_points = [p for s in strokes for p in s]
    if not all_points: return strokes
    arr = np.array([np.asarray(p[:2], dtype=np.float32) for p in all_points])
    min_xy, max_xy = arr.min(0), arr.max(0)
    scale = max(max_xy[0] - min_xy[0], max_xy[1] - min_xy[1], 1e-6)

    out = []
    for s in strokes:
        new_s = []
        for p in s:
            extra = tuple(p[2:]) if len(p) > 2 else tuple()
            x = (p[0] - min_xy[0]) / scale * coord_range
            y = (p[1] - min_xy[1]) / scale * coord_range
            new_s.append((x, y) + extra)
        out.append(new_s)
    return out

def simplify_strokes(strokes, dist_thresh=5, angle_thresh=0.15):
    out = []
    for s in strokes:
        if len(s) <= 2:
            out.append(s); continue
        simp = [s[0]]
        for i in range(1, len(s) - 1):
            prev, curr, next_p = simp[-1], s[i], s[i + 1]
            d1 = (curr[0] - prev[0], curr[1] - prev[1])
            d2 = (next_p[0] - curr[0], next_p[1] - curr[1])
            dist_sq = d1[0] ** 2 + d1[1] ** 2
            if d1 == (0, 0) or d2 == (0, 0): ang = 0.0
            else: ang = abs(math.atan2(d2[1], d2[0]) - math.atan2(d1[1], d1[0]))
            if dist_sq > dist_thresh ** 2 or ang > angle_thresh:
                simp.append(curr)
        simp.append(s[-1])
        out.append(simp)
    return out

def speed_normalization(strokes, target_speed=10.0):
    out = []
    for s in strokes:
        if len(s) <= 1:
            out.append(s); continue
        new_s = [s[0]]
        acc_dist = 0.0
        p1 = s[0]
        for i in range(1, len(s)):
            p2 = s[i]
            dist = math.hypot(p2[0] - p1[0], p2[1] - p1[1])
            acc_dist += dist
            curr_p1 = p1
            curr_dist = dist
            while acc_dist >= target_speed:
                acc_dist -= target_speed
                if curr_dist > 1e-6:
                    t = 1 - (acc_dist / curr_dist)
                    extra = tuple(curr_p1[2:]) if len(curr_p1) > 2 else tuple()
                    new_pt = (curr_p1[0] + t * (p2[0] - curr_p1[0]), curr_p1[1] + t * (p2[1] - curr_p1[1])) + extra
                    new_s.append(new_pt)
                    curr_p1 = new_pt
                    curr_dist = acc_dist
                else: break
            p1 = p2
        new_s.append(s[-1])
        out.append(new_s)
    return out

def add_imaginary_stroke(strokes, add_ink_dim=True, add_pen_states=False):
    if len(strokes) < 2: return [{"data": np.array(s, dtype=np.float32), "type": "real"} for s in strokes]
    total_len, count = 0.0, 0
    for s in strokes:
        for i in range(len(s) - 1):
            total_len += math.hypot(s[i + 1][0] - s[i][0], s[i + 1][1] - s[i][1])
            count += 1
    avg_len = total_len / max(1, count)

    out = []
    for i in range(len(strokes)):
        out.append({"data": np.array(strokes[i], dtype=np.float32), "type": "real"})
        if i < len(strokes) - 1:
            start = strokes[i][-1]
            end = strokes[i + 1][0]
            dist = math.hypot(end[0] - start[0], end[1] - start[1])
            steps = max(1, int(math.ceil(dist / max(1e-6, avg_len))))
            v_stroke = []
            for j in range(steps + 1):
                alpha = j / steps
                extra = tuple(start[2:]) if len(start) > 2 else tuple()
                pt = (start[0] * (1 - alpha) + end[0] * alpha, start[1] * (1 - alpha) + end[1] * alpha) + extra
                v_stroke.append(pt)
            out.append({"data": np.array(v_stroke, dtype=np.float32), "type": "virtual"})
    return out

def apply_data_augmentation(strokes, rng):
    if not strokes: return strokes
    sx = 1.0 + rng.uniform(-0.15, 0.15)
    sy = 1.0 + rng.uniform(-0.15, 0.15)
    tx = rng.normal(0, 1) * 10.0
    ty = rng.normal(0, 1) * 10.0
    eps = rng.uniform(0.0, 2.0)

    all_pts = []
    for s in strokes:
        for p in s: all_pts.append(p[:2])
    if len(all_pts) == 0: return strokes
    arr = np.array(all_pts, dtype=np.float32)
    w, h = (arr.max(0) - arr.min(0)) + 1e-6
    fx, fy = 2 * math.pi / h, 2 * math.pi / w

    aug_strokes = []
    for s in strokes:
        new_stroke = []
        for p in s:
            x = p[0] * sx + tx + eps * math.sin(p[1] * sy * fx)
            y = p[1] * sy + ty + eps * math.sin(p[0] * sx * fy)
            new_pt = (x, y) + tuple(p[2:])
            new_stroke.append(new_pt)
        aug_strokes.append(new_stroke)
    return aug_strokes

def apply_rotation_on_strokes(strokes, angle_deg):
    if angle_deg is None or angle_deg == 0: return strokes
    theta = np.deg2rad(float(angle_deg))
    c, s = np.cos(theta), np.sin(theta)
    out = []
    for stroke in strokes:
        s_arr = np.array(stroke, dtype=np.float32)
        if len(s_arr) == 0:
            out.append(s_arr); continue
        x, y = s_arr[:, 0], s_arr[:, 1]
        xr = c * x - s * y
        yr = s * x + c * y
        if s_arr.shape[1] > 2: new_s = np.concatenate([np.stack([xr, yr], 1), s_arr[:, 2:]], 1)
        else: new_s = np.stack([xr, yr], 1)
        out.append([tuple(row) for row in np.asarray(new_s)])
    return out

def build_feature_sequence(input_strokes, add_ink_dim, add_pen_states, use_derivatives):
    if len(input_strokes) > 0 and isinstance(input_strokes[0], dict):
        stroke_objs = input_strokes
    else:
        stroke_objs = [{"data": np.array(s, dtype=np.float32), "type": "real"} for s in input_strokes]

    processed = []
    for item in stroke_objs:
        s, stype = item["data"], item["type"]
        xy = s[:, :2]
        if add_ink_dim:
            if stype == 'real':
                if s.shape[1] >= 3: ink = s[:, 2:3]
                else: ink = np.ones((len(s), 1), dtype=np.float32)
            else: ink = np.zeros((len(s), 1), dtype=np.float32)
            xy = np.concatenate([xy, ink], axis=1)
        if add_pen_states:
            pd = np.zeros((len(s), 1), dtype=np.float32)
            pu = np.zeros((len(s), 1), dtype=np.float32)
            if stype == 'real' and len(s) > 0: pd[0] = 1.0; pu[-1] = 1.0
            xy = np.concatenate([xy, pd, pu], axis=1)
        processed.append(xy)

    if not processed: return np.zeros((0, 2), dtype=np.float32)
    
    if use_derivatives:
        final_seq = []
        for feat in processed:
            coords = feat[:, :2]
            n = len(coords)
            vd = np.zeros_like(coords)
            vc = np.zeros_like(coords)
            if n >= 2:
                vd[0] = coords[1] - coords[0]
                vd[-1] = coords[-1] - coords[-2]
                if n > 2: vd[1:-1] = (coords[2:] - coords[:-2]) / 2.0
                vc[0] = vd[1] - vd[0]
                vc[-1] = vd[-1] - vd[-2]
                if n > 2: vc[1:-1] = (vd[2:] - vd[:-2]) / 2.0
            final_seq.append(np.concatenate([feat, vd, vc], axis=1))
        return np.concatenate(final_seq, 0)
    return np.concatenate(processed, 0)

def normalize_all_dimensions(sequence, add_ink_dim, add_pen_states):
    if len(sequence) == 0: return sequence
    seq = sequence.copy()
    current_idx = 2
    if add_ink_dim:
        max_ink = seq[:, 2].max()
        if max_ink > 0: seq[:, 2] /= max_ink
        current_idx += 1
    if add_pen_states: current_idx += 2
    if current_idx < seq.shape[1]:
        derivs = seq[:, current_idx:]
        max_val = np.max(np.abs(derivs))
        if max_val > 1e-6: seq[:, current_idx:] /= max_val
    return seq

def process_sequence_length(seq, target_len, use_end_pad):
    L, D = seq.shape
    if L >= target_len: return seq[:target_len]
    out = np.zeros((target_len, D), dtype=np.float32)
    out[:L] = seq
    if use_end_pad and L > 0:
        out[L:] = seq[-1]
        if D >= 4: out[L:, -4:] = 0
    return out

def compute_window_signatures(path, window_size=10, stride=1, depth=2):
    length, dim = path.shape
    if length < window_size: window_size = length
    num_windows = max(1, (length - window_size) // stride + 1)
    
    starts = np.arange(num_windows) * stride
    indices = starts[:, None] + np.arange(window_size)[None, :]
    indices = np.clip(indices, 0, length - 1)
    
    windows = path[indices]
    sigs = iisignature.sig(windows, depth)
    return sigs

def build_one_feature(strokes, cfg, rng_aug=None, angle=None):
    if cfg.get('use_simplify', False):
        strokes = simplify_strokes(strokes, cfg.get('dist_thresh', 5), cfg.get('angle_thresh', 0.15))
    if cfg.get('use_speed_norm', False):
        strokes = speed_normalization(strokes, cfg.get('target_speed', 10.0))
    if angle:
        strokes = apply_rotation_on_strokes(strokes, angle)
    if rng_aug is not None and cfg.get('use_data_augmentation', False):
        strokes = apply_data_augmentation(strokes, rng_aug)
    if cfg.get('use_hanging', False):
        strokes = apply_hanging_on_strokes(strokes, cfg.get('hanging_mode', "SC"))

    strokes = normalize_strokes(strokes, cfg.get('coord_range', 1.0))
    if cfg.get('add_imaginary_stroke', False) and len(strokes) > 1:
        strokes_data = add_imaginary_stroke(strokes, cfg.get('add_ink_dim', False), cfg.get('add_pen_states', False))
    else:
        strokes_data = [{"data": np.array(s, dtype=np.float32), "type": "real"} for s in strokes]

    seq = build_feature_sequence(strokes_data, cfg.get('add_ink_dim', False), cfg.get('add_pen_states', False), cfg.get('use_derivatives', False))
    seq = normalize_all_dimensions(seq, cfg.get('add_ink_dim', False), cfg.get('add_pen_states', False))
    seq = process_sequence_length(seq, cfg.get('target_length', 100), cfg.get('use_end_padding', True))

    if cfg.get('use_signatures', False):
        return compute_window_signatures(seq, cfg.get('window_size', 5), cfg.get('stride', 1), cfg.get('depth', 2))
    return seq

def process_class(c_idx, data_dir, is_train, cfg, seed=None):
    raw, _ = read_pot_file(os.path.join(data_dir, f"{c_idx}.pot"))
    if not raw:
        if is_train: 
            return [], [], []
        else: 
            return [], [], []

    n_tr = int(0.8 * len(raw))
    sel = raw[:n_tr] if is_train else raw[n_tr:]
    
    rng = np.random.RandomState(int(seed) if seed is not None else 42)

    if is_train:
        feats, labels, angles = [], [], []
        for i, s in enumerate(sel):
            ang = rng.uniform(0, 360)
            f = build_one_feature(s, cfg, rng_aug=rng, angle=ang)
            feats.append(f)
            labels.append(c_idx - 1)
            angles.append(ang)
        return feats, labels, angles
    else:
        f30, l30, a30 = [], [], []
        ang_list = list(range(0, 360, 12))
        for i, s in enumerate(sel):
            for ang in ang_list:
                f30.append(build_one_feature(s, cfg, angle=float(ang)))
                l30.append(c_idx - 1)
                a30.append(float(ang))
        return f30, l30, a30


class TorchDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(np.array(X), dtype=torch.float32)
        self.y = torch.tensor(np.array(y), dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def make_loaders(data_dir, n_cls, cfg, seed=42, is_parallel=True):
    tr, te30 = ([], [], []), ([], [], [])
    
    if is_parallel:
        results = Parallel(n_jobs=-1)(
            delayed(process_class)(c, data_dir, True, cfg, seed)
            for c in tqdm(range(1, n_cls + 1), desc="Build train/test set")
        )
        for res in results:
            F, L, A = res
            tr[0].extend(F)
            tr[1].extend(L)
            tr[2].extend(A)
        
        # 生成测试集（30个角度）
        results_test = Parallel(n_jobs=-1)(
            delayed(process_class)(c, data_dir, False, cfg, seed)
            for c in tqdm(range(1, n_cls + 1), desc="Build test set")
        )
        for res in results_test:
            f30, l30, a30 = res
            te30[0].extend(f30)
            te30[1].extend(l30)
            te30[2].extend(a30)
    else:
        for c in tqdm(range(1, n_cls + 1), desc="Build train/test set"):
            F, L, A = process_class(c, data_dir, True, cfg, seed)
            tr[0].extend(F)
            tr[1].extend(L)
            tr[2].extend(A)
        
        for c in tqdm(range(1, n_cls + 1), desc="Build test set"):
            f30, l30, a30 = process_class(c, data_dir, False, cfg, seed)
            te30[0].extend(f30)
            te30[1].extend(l30)
            te30[2].extend(a30)

    bs = cfg.get('batch_size', 64)

    tr_dataset = TorchDataset(tr[0], tr[1])
    tr_loader = DataLoader(tr_dataset, batch_size=bs, shuffle=True, drop_last=False)

    te30_dataset = TorchDataset(te30[0], te30[1])
    te30_loader = DataLoader(te30_dataset, batch_size=512, shuffle=False)

    feat_dim = tr_dataset.X.shape[-1] if len(tr_dataset.X) > 0 else None
    print(f"Train samples: {len(tr[0])}, shape: {tr_dataset.X.shape}")
    print(f"Test30 samples: {len(te30[0])}, shape: {te30_dataset.X.shape}")
    print(f"Feature dimension: {feat_dim}")
    
    return tr_loader, te30_loader, feat_dim


In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class TransformerClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=256, nhead=8, num_layers=6, dim_feedforward=1024, dropout=0.3, max_len=1000):
        super(TransformerClassifier, self).__init__()
        
        self.d_model = d_model
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout, 
            batch_first=True,
            norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.input_proj(x)
        x = x * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        out = self.fc_out(x)
        return out


In [ ]:
import torch
from thop import profile
from thop import clever_format

def get_model_metrics(model, seq_len, input_dim):
    batch_size = 1
    device = next(model.parameters()).device
    dummy_input = torch.randn(batch_size, seq_len, input_dim).to(device)
    
    macs, params = profile(model, inputs=(dummy_input, ), verbose=False)
    flops = macs * 2.0
    flops_str, params_str = clever_format([flops, params], "%.3f")
    
    return params_str, flops_str


def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0
    steps = 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)
        steps += 1
    return total_loss / steps, total_correct / total_samples, steps

def evaluate(model, dataloader):
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            logits = model(x)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(y.numpy())
    all_logits = np.concatenate(all_logits, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    acc = np.mean(np.argmax(all_logits, axis=-1) == all_targets)
    return all_logits, all_targets, acc

def train_loop(model, tr_loader, te30_loader, cfg, epochs, save_dir):
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=cfg['lambd'])
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
    criterion = nn.CrossEntropyLoss()

    log = {"epoch": [], "loss": [], "acc": [], "test_acc": [], "time": [], "mem": []}
    total_steps, t0_total = 0, time.time()

    for ep in range(epochs):
        t0_epoch = time.time()

        l, a, steps_in_epoch = train_epoch(model, tr_loader, optimizer, criterion)
        scheduler.step()
        total_steps += steps_in_epoch

        _, _, test_acc = evaluate(model, te30_loader)
        test_acc *= 100

        cur_mem = get_gpu_memory_usage_mb()
        epoch_time = time.time() - t0_epoch

        if (ep + 1) % cfg['print_every'] == 0 or ep == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"Epoch {ep+1}/{epochs} | Loss: {l:.4f} | Train Acc: {a:.2%} | Test Acc: {test_acc:.2f}% | Mem: {cur_mem:.2f}MB | Time: {epoch_time:.2f}s")

        log["epoch"].append(ep + 1)
        log["loss"].append(l)
        log["acc"].append(a * 100)
        log["test_acc"].append(test_acc)
        log["time"].append(epoch_time)
        log["mem"].append(cur_mem)

        if save_dir and cfg.get('save_ckpt_every', 0) and (ep + 1) % cfg['save_ckpt_every'] == 0:
            os.makedirs(os.path.join(save_dir, "checkpoints"), exist_ok=True)
            torch.save(model.state_dict(), os.path.join(save_dir, "checkpoints", f"ep{ep+1}.pth"))

    total_time = time.time() - t0_total
    avg_mem = np.mean(log["mem"]) if log["mem"] else 0.0
    time_per_1000_steps = (total_time / total_steps * 1000) if total_steps > 0 else 0.0

    stats = {
        "total_time": total_time,
        "total_steps": total_steps,
        "avg_mem": avg_mem,
        "time_per_1000_steps": time_per_1000_steps
    }
    return model, log, stats

def run_single_model(tr_load, te30_load, data_dim, n_cls, save_dir, cfg, seed):
    os.makedirs(save_dir, exist_ok=True)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = TransformerClassifier(
        input_dim=data_dim,
        num_classes=n_cls,
        d_model=cfg['hidden_dim'],
        nhead=cfg['num_heads'],
        num_layers=cfg['num_blocks'],
        dim_feedforward=cfg['d_ff'],
        dropout=cfg['drop_rate']
    ).to(device)

    seq_len = cfg.get('target_length', 100) 
    params_str, flops_str = get_model_metrics(model, seq_len=seq_len, input_dim=data_dim)
    
    print(f"\n[模型分析] Transformer 参数量: {params_str}, FLOPs: {flops_str}")

    print(f"\n>>> 开始训练 Transformer (Seed {seed})...")
    model, log, stats = train_loop(model, tr_load, te30_load, cfg, cfg['total_epochs'], save_dir)

    epochs_to_save = min(100, len(log['test_acc']))
    if epochs_to_save > 0:
        df_acc = pd.DataFrame({
            "Epoch": log['epoch'][:epochs_to_save],
            "Test_Accuracy": log['test_acc'][:epochs_to_save]
        })
        csv_path = os.path.join(save_dir, f"test_acc_first_100_epochs_seed_{seed}.csv")
        df_acc.to_csv(csv_path, index=False)
        print(f"前 {epochs_to_save} 个 Epoch 的测试精度已保存至: {csv_path}")

    l30, t30, a30 = evaluate(model, te30_load)
    print(f"[Seed {seed}] Final Test Accuracy (30x angles): {a30*100:.2f}%")
    print(f"Time per 1000 steps: {stats['time_per_1000_steps']:.2f}s")
    print(f"Avg GPU Memory: {stats['avg_mem']:.2f}MB")
    print(f"Transformer 参数量: {params_str}, FLOPs: {flops_str}")

    torch.save(model.state_dict(), os.path.join(save_dir, "model.pth"))

    res_summary = {
        "seed": seed,
        "acc": float(a30),
        "train_time": stats['total_time'],
        "total_steps": stats['total_steps'],
        "time_per_1000_steps": stats['time_per_1000_steps'],
        "avg_mem": stats['avg_mem'],
        "num_params": params_str, 
        "flops": flops_str
    }

    with open(os.path.join(save_dir, "summary.json"), "w") as f:
        json.dump(res_summary, f, indent=2)

    return {
        "summary": res_summary,
        "res_30x": {"logits": l30, "targets": t30, "acc": a30},
    }

def compute_ensemble_metrics(results_list):
    if not results_list:
        return {}
    accs = [r['acc'] for r in results_list]
    mean_acc, std_acc = np.mean(accs), np.std(accs)
    all_logits = np.stack([r['logits'] for r in results_list], axis=0)
    targets = results_list[0]['targets']

    probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)
    avg_probs = np.mean(probs, axis=0)
    soft_acc = np.mean(np.argmax(avg_probs, -1) == targets)

    seed_preds = np.argmax(all_logits, axis=-1)
    hard_preds = []
    for i in range(seed_preds.shape[1]):
        counts = np.bincount(seed_preds[:, i])
        hard_preds.append(np.argmax(counts))
    hard_acc = np.mean(np.array(hard_preds) == targets)

    return {"mean": mean_acc, "std": std_acc, "soft": soft_acc, "hard": hard_acc}

def run_experiment(data_dir, n_cls, save_dir, cfg):
    seeds = cfg.pop('seed_list')
    print(f">>> Training with Seeds: {seeds}")
    
    cfg['use_hanging'] = True
    tr_load, te30_load, data_dim = make_loaders(
        data_dir, n_cls, cfg, seed=42,  is_parallel=True
    )
    
    results = []
    for s in seeds:
        print(f"\n{'='*60}\nTraining with Seed {s}\n{'='*60}")
        res = run_single_model(
            tr_load, te30_load, data_dim, n_cls,
            os.path.join(save_dir, f"seed_{s}"), 
            cfg.copy(), s
        )
        if res:
            results.append(res)
            print(f"\nSeed {s} Results: Acc={res['summary']['acc']*100:.2f}%")
    
    if results:
        m30 = compute_ensemble_metrics([r['res_30x'] for r in results])
        print("\n" + "="*60 + "\nFINAL RESULTS\n" + "="*60)
        print(f"Test 30x (Rotated): Mean ± Std: {m30['mean']*100:.2f}% ± {m30['std']*100:.2f}%")
        print(f"Soft Voting: {m30['soft']*100:.2f}% | Hard Voting: {m30['hard']*100:.2f}%")
        print("="*60)


if __name__ == "__main__":
    DATA_DIR = "RotFreeData/Radicals52"
    SAVE_DIR = "Model/Transformer/Transformer_Radicals52"
    NUM_CLASSES = 52
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    BASE_CONFIG = {
        "use_simplify": False, "dist_thresh": 5.0, "angle_thresh": 0.15,
        "use_speed_norm": True, "target_speed": 10.0, "coord_range": 1.0,
        "add_ink_dim": False, "add_pen_states": False, "add_imaginary_stroke": False,
        "use_derivatives": False, "use_end_padding": True,
        "use_signatures": False,
        "use_data_augmentation": False,
        "target_length": 100, "window_size": 5, "stride": 1, "depth": 2,
        "use_hanging": False, "hanging_mode": "SC",

        "print_every": 1, "save_ckpt_every": 0, "total_epochs": 300, "batch_size": 64,
        "seed_list": [1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020],

        "num_blocks": 6,
        "hidden_dim": 256,
        "num_heads": 8,
        "d_ff": 1024,
        "drop_rate": 0.3,
        "lambd": 1e-4
    }

    run_experiment(DATA_DIR, NUM_CLASSES, SAVE_DIR, BASE_CONFIG)
